# 02 · Pair selection

Choose the pair to trade, using **TRAIN data only**, and record it in `config.yaml`.

Plan: correlation screen → Engle-Granger cointegration test of every pair → sanity checks
(multiple testing, single-leg stationarity, stability over time) → look at the spreads → decide.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Make the repo root importable, whether Jupyter starts in notebooks/ or the repo root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import data, strategy, backtest, plots

plots.apply_style()
pd.set_option("display.float_format", "{:.4f}".format)
cfg = data.load_config()

## 1. Load train data

In [ ]:
prices = data.load_prices(cfg, verbose=False)
funding = data.load_funding(cfg, coins=list(prices.columns))
train_prices, _ = data.split_train_test(prices, cfg["setup"]["train_end"])  # the test part is not used here
print(f"Train: {train_prices.index[0]} to {train_prices.index[-1]} ({len(train_prices)} hourly bars, {train_prices.shape[1]} coins)")

## 2. Correlation screen

Correlation of hourly log returns is a fast first filter: a pair whose returns barely move together is
unlikely to have a stable spread. But high correlation is **not** cointegration: two coins can be highly
correlated hour by hour and still drift apart over months.

In [ ]:
corr = np.log(train_prices).diff().corr()
plots.plot_correlation_matrix(corr, title="Correlation of hourly log returns (train period)")

In [ ]:
strategy.correlation_screen(train_prices, top_n=10)

## 3. Cointegration ranking

`strategy.rank_pairs` runs for every pair (both directions, keeps the better one):

- **Engle-Granger test**: OLS of log(A) on log(B) gives alpha and beta; an ADF test on the residual spread gives the p-value (with Engle-Granger critical values).
- **half_life** (hours): how fast the spread halves its distance to the mean, from an AR(1) regression.
- **stability**: share of rolling 180-day windows (daily data, step 30 days) in which the pair is cointegrated at 5%.
- **p_bonferroni**: p-value × number of tests (see the next cell).

This takes about a minute.

In [ ]:
ranking = strategy.rank_pairs(train_prices)
top10 = ranking.head(10)
top10.style.format({"correlation": "{:.2f}", "p_value": "{:.4f}", "p_bonferroni": "{:.3f}", "adf_stat": "{:.2f}",
                    "beta": "{:.3f}", "half_life": "{:.0f}", "stability": "{:.0%}"})

In [ ]:
plots.plot_pair_ranking(ranking, top_n=10, save="02_pair_ranking.png")

### The multiple-testing problem

We tested 66 pairs in 2 directions, i.e. **132 hypothesis tests**. At a 5% level, about 6–7 pairs would look
"cointegrated" **by pure chance** even if no real relationship existed. The smallest p-value of 132 tests is
therefore much less impressive than the same p-value from a single, pre-planned test.

The **Bonferroni correction** multiplies each p-value by the number of tests. It is conservative (it assumes
the tests are independent, while our pairs share coins), but it makes the point: a pair whose
`p_bonferroni` is far above 0.05 is not convincingly cointegrated. That is why we do not choose by p-value
alone, but also look at stability over time, half-life and economic sense.

## 4. Sanity check: is one leg stationary on its own?

If one coin's log price is already mean-reverting by itself (e.g. it traded in a range for the whole train
period), the regression finds a small beta and the "spread" is essentially that coin alone. The pair then looks
cointegrated although the other leg adds nothing, and the relationship breaks as soon as that coin trends.
A genuine pair has two **non-stationary** legs (high p-values below) and a stationary spread.

In [ ]:
adf = strategy.adf_levels(train_prices).sort_values().to_frame()
adf["stationary_alone (p < 0.05)"] = adf["adf_p_value_levels"] < 0.05
adf

**What we see (train data, at the time of writing):** LTC is the only coin whose own log price is stationary
(ADF p ≈ 0.01). This explains why the top of the ranking is LTC paired with almost everything, always with a
small beta (≈ 0.05–0.3). These are LTC-range effects rather than pair relationships, so treat them with care.

## 5. Spreads of the top 3 pairs (and the ETH/BTC placeholder)

A first look with the **current config parameters** (rolling hedge ratio, z-score, entry/exit thresholds).
This is only to see whether the spread oscillates around its mean; the parameters are calibrated in notebook 03.
`half_life_h` here is the half-life of the spread we actually trade (rolling hedge ratio), which is the one
that matters for the time stop.

In [ ]:
params = data.get_params(cfg)
candidates = [tuple(pair) for pair in ranking[["a", "b"]].head(3).to_numpy()] + [("ETH", "BTC")]

first_look = {}
for pair in candidates:
    result = backtest.run_strategy(prices, funding, pair, params, period="train")
    first_look[f"{pair[0]}/{pair[1]}"] = result
    display(plots.plot_spread_zscore(result))

In [ ]:
summary = backtest.compare_metrics(first_look).loc[["sharpe", "total_return", "max_drawdown", "n_trades", "win_rate"]]
summary.loc["half_life_h"] = [res.params["half_life"] for res in first_look.values()]
summary

## 6. Stability over time

Rolling Engle-Granger p-value on daily closes (180-day window, moved forward 7 days at a time). A good pair stays
below the 5% line (dashed) most of the time; a pair that dips below only once was a one-off.

In [ ]:
daily = train_prices.resample("1D").last()
rolling_p = {f"{a}/{b}": strategy.rolling_coint_pvalues(daily[a], daily[b], window=180, step=7)
             for a, b in candidates}
plots.plot_series(rolling_p, title="Rolling cointegration p-value (180-day window, daily data, train)",
                  ylabel="p-value", hline=0.05, save="02_rolling_pvalues.png")

## 7. What makes a good pair

A pair is a good candidate if it is:

1. **Cointegrated**: low p-value, ideally still small after the multiple-testing correction.
2. **Stable**: cointegrated in a large share of rolling windows, not just once (stability column, rolling plot).
3. **Fast enough**: a half-life between roughly **6 hours and 2 weeks**. Faster than that and costs eat every
   trade; slower and we hold for weeks, pay funding and rarely trade (few trades = weak statistics).
4. **Economically sensible**: there is a reason for the two coins to move together (e.g. both large-cap
   smart-contract platforms, same sector, similar investor base). A purely statistical pair tends to break out of sample.
5. **Not driven by one stationary leg** (section 4).

No pair will tick every box (crypto pairs rarely do over three years); the decision is a trade-off that we
must be able to justify in the report.

> ## 🟡 TEAM DECISION: choose the pair
>
> **Look at:** the ranking table (p-value, p_bonferroni, half-life, stability), the single-leg check (section 4),
> the spread plots and the rolling p-values.
>
> **How to choose:** prefer a pair that is reasonably cointegrated **and** stable, with a half-life in the
> 6h–2 weeks range of the traded spread (`half_life_h` in section 5), and a story you can explain.
> Do **not** pick the pair with the best Sharpe in section 5: that is overfitting the pair choice.
> Order matters: `[A, B]` means the spread is log(A) − beta·log(B).
>
> Write it down in one paragraph (why this pair, what the alternatives were), then save it with the call below.

In [ ]:
# Uncomment and edit after the team has decided, then rerun this notebook from the top:
# data.update_config({"pair": ["ETH", "BTC"]})